In [11]:
import random
from typing import Callable, Tuple

import torch
import torchvision
from torch import Tensor, nn, optim
from torch.nn import functional as F
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import STL10
from torchvision.transforms import ToTensor

In [12]:
class RandomApply(nn.Module):
    def __init__(self, fn: Callable, p: float):
        super().__init__()
        self.fn = fn
        self.p = p

    def forward(self, x: Tensor) -> Tensor:
        if random.random() < self.p:
            x = self.fn(x)
        return x


def default_augmentation(image_size: Tuple[int, int] = (96, 96)):
    return T.Compose([
        RandomApply(T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1), p=0.8),
        RandomApply(T.Grayscale(num_output_channels=3), p=0.2),
        RandomApply(T.RandomHorizontalFlip(p=0.5), p=0.8),
        RandomApply(T.GaussianBlur(kernel_size=(3, 3), sigma=(1.5, 1.5)), p=0.2),
        RandomApply(T.RandomResizedCrop(image_size, scale=(0.08, 1.0), ratio=(0.75, 1.333)), p=0.5),
        T.Normalize(
            mean=torch.tensor([0.485, 0.456, 0.406]),
            std=torch.tensor([0.229, 0.224, 0.225]),
        ),
    ])


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [14]:
def get_encoder_model(**kwargs):
    resnet = torchvision.models.resnet18(**kwargs)
    return torch.nn.Sequential(*list(resnet.children())[:-1])

In [15]:
class NormalizedMSELoss(nn.Module):
    def __init__(self) -> None:
        super(NormalizedMSELoss,self).__init__()

    def forward(self, view1: Tensor, view2: Tensor) -> Tensor:
        v1 = F.normalize(view1, dim=-1)
        v2 = F.normalize(view2, dim=-1)
        return 2 - 2 * (v1 * v2).sum(dim=-1)

In [16]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, projection_dim: int=128, hidden_dim: int=512) -> None:
        super(MLP,self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, projection_dim)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [17]:
class EncoderProjecter(nn.Module):
    def __init__(self,
                 encoder: nn.Module,
                 hidden_dim: int=512,
                 projection_out_dim: int=128
                 ) -> None:
        super(EncoderProjecter, self).__init__()

        self.encoder = encoder
        self.projetion = MLP(input_dim=512, projection_dim=projection_out_dim, hidden_dim=hidden_dim)

    def forward(self, x):
        h = self.encoder(x)
        h = h.view(h.shape[0], h.shape[1])
        return self.projetion(h)

In [18]:
class BYOL(nn.Module):
    def __init__(self,
                 hidden_dim: int = 512,
                 projection_out_dim: int = 128,
                 target_decay: float = 0.996
                ) -> None:
        super(BYOL, self).__init__()
        resnet = get_encoder_model()
        self.online_network = EncoderProjecter(resnet)  # encoder + projector
        self.online_predictor = MLP(input_dim=projection_out_dim, projection_dim=projection_out_dim, hidden_dim=hidden_dim)

        self.target_network = EncoderProjecter(resnet)  # init with copy of parameters of online network
        self.target_network.load_state_dict(self.online_network.state_dict())
        
        # set target_network's weights to be untrainable
        self.target_network.eval()
        for param in self.target_network.parameters():
            param.requires_grad = False
        self.target_decay = target_decay
        self.loss_function = NormalizedMSELoss()


    @torch.no_grad()
    def soft_update_target_network(self) -> None:
        for online_p, target_p in zip(self.online_network.parameters(), self.target_network.parameters()):
            target_p.data = target_p.data * self.target_decay + online_p.data * (1. - self.target_decay)


    def forward(self, view) -> Tuple[Tensor]:
        online_proj = self.online_network(view)
        target_proj = self.target_network(view)

        return online_proj, target_proj

    def loss(self, view1, view2):
        online_proj1, target_proj1 = self(view1)
        online_proj2, target_proj2 = self(view2)

        online_prediction_1 = self.online_predictor(online_proj1)
        online_prediction_2 = self.online_predictor(online_proj2)

        loss1 = self.loss_function(online_prediction_1, target_proj2.detach())
        loss2 = self.loss_function(online_prediction_2, target_proj1.detach())
        return torch.mean(loss1 + loss2)

In [19]:
TRAIN_DATASET = STL10(root="data", split="train", download=True, transform=ToTensor())
TRAIN_UNLABELED_DATASET = STL10(root="data", split="train+unlabeled", download=True, transform=ToTensor())
TEST_DATASET = STL10(root="data", split="test", download=True, transform=ToTensor())

100%|██████████████████████████████████████████████████████████████████████████████| 2.64G/2.64G [09:55<00:00, 4.43MB/s]


Extracting data/stl10_binary.tar.gz to data
Files already downloaded and verified
Files already downloaded and verified


In [20]:
LABELED_BATCH_SIZE = 128
UNLABELED_BATCH_SIZE = 1024

train_dataloader = DataLoader(TRAIN_DATASET, batch_size=LABELED_BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
train_unlabeled_dataloader = DataLoader(TRAIN_UNLABELED_DATASET, batch_size=UNLABELED_BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_dataloader = DataLoader(TEST_DATASET, batch_size=LABELED_BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [3]:
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

In [4]:
learner = BYOL(
    resnet,
    image_size = 256,
    hidden_layer = 'avgpool'
)

In [5]:
opt = torch.optim.Adam(learner.parameters(), lr=3e-4)

In [6]:
import os
from PIL import Image
from torch.utils.data import Dataset 

In [7]:
class TileDatasetTrain(Dataset):
    def __init__(self, folder_path, online_transform=None, target_transform=None):
        self.folder_path = folder_path
        self.image_files = [
            os.path.join(root, f) 
            for root, _, files in os.walk(folder_path) 
            for f in files if f.endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))
        ]
        self.online_transform = online_transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        img = Image.open(img_path).convert("RGB")
        
        if self.online_transform and self.target_transform:
            online_img = self.online_transform(img)
            target_img = self.target_transform(img)
            return online_img
        
        return img

In [8]:
online_transforms = transforms.Compose([ 
        transforms.Resize((256, 256)), 
        transforms.RandomRotation(degrees=(-45, 45)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(), 
        transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
            
        # Color augmentations
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), 
            
        # Noise and blurring
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Second set of augmentations (for target network)
target_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomRotation(degrees=(-45, 45)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(), 
        transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
            
        # Color augmentations
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), 
            
        # Noise and blurring
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]) 

In [10]:
print("\nLoading training data...")
dataset1 = TileDatasetTrain("/data_14T/Raja/Pre_processing_test/RGB", 
                                     online_transform=online_transforms,
                                     target_transform=target_transforms)
print("\nNumber of training samples:", len(dataset1))


Loading training data...

Number of training samples: 1870


In [ ]:

data_loader = DataLoader(dataset1, batch_size=config.batch_size_train, 
                           num_workers=config.num_workers, shuffle=True)